In [1]:
import pdfplumber
import csv
import pandas as pd

In [2]:
PATH_TO_INPUT_PDF = '../scripts/skript_005.pdf'
PATH_TO_OUTPUT_CSV = '../csv/005.csv'

# Create a list of Speaker/Roles

The following dataframe manipulation was done by chatgpt on 28-01-2026 with GPT-5-mini and this prompt
> i have a dataframe with a string column 'names' these strings have comma (,) inside them i want to split these strings by the comma and have the before and after as a single entry i after that want to split the resulting strings by whitespace at the end i want the original and the seperated values in an array

In [65]:
def split_names(s):
    # 1. Original string
    result = [s]
    
    # 2. Split by comma and strip
    parts = [part.strip() for part in s.split(",")]
    
    # 3. Include the comma-separated chunks
    result.extend(parts)
    
    # 4. Split each chunk by whitespace and flatten
    words = [word for part in parts for word in part.split()]
    
    # 5. Add individual words
    result.extend(words)
    
    return result

speakers = pd.read_csv('../data/rolle.tsv', sep='\t')

# Apply function
speakers['name'] = speakers['name'].apply(split_names)
speakers = speakers.explode('name')
speakers = speakers.drop(columns='rolleID')
additional_speakers = ['Tommy', 'Justua', 'Schwarzbart', 'Allgemein', 'Mrs.Peterson', 'DwiggIns', '3???/ Gus', '3???']
speakers = pd.concat([speakers, pd.Series(additional_speakers, name="name")], ignore_index=True)
speakers = speakers.reset_index(drop=True)
speakers

,name
0,"Hitchcock, Erzähler"
1,Hitchcock
2,Erzähler
3,Hitchcock
4,Erzähler
...,...
9179,Allgemein
9180,Mrs.Peterson
9181,DwiggIns
9182,3???/ Gus


In [66]:
speakers_lookup_set = set(speakers['name'].values)
def lookup_speaker_name(text: str):
    words = text.strip().split(' ')
    #print(words)
    matches = []
    for i in range(len(words)):
        #print(i)
        search_string = ' '.join(words[0:i+1])
        #print(search_string)
        if search_string in speakers_lookup_set:
            # append found word and last index of word in string
            matches.append((search_string, text.find(search_string) + len(search_string) - 1))
    if len(matches) == 0:
        return None, 0
    return max(matches, key=lambda x: len(x[0]))

In [67]:
# test lookup
texts = ['Bob Andrews bla bla bla', 'Justus Jonas bli bla blup', '3???/ Gus Danke!']
for text in texts:
    print(text, lookup_speaker_name(text))

Bob Andrews bla bla bla ('Bob Andrews', 10)
Justus Jonas bli bla blup ('Justus Jonas', 11)
3???/ Gus Danke! ('3???/ Gus', 8)


# Simple Text Extractor using pdfplumber
- extracts text while keeping layout
- iterates through text to find the layout
- needs manual settings for table layout

## Import and load PDF

In [47]:
pdf = pdfplumber.open(PATH_TO_INPUT_PDF)

## Set Speaker Name Window

In [48]:
# SETTINGS
SPEAKER_NAME_WINDOW = [9, 23]

#SPEAKER_NAME_WINDOW = [5, 18]
#         |           WINDOW        |
#_._._._._|_._.J.U.S.T.U.S._._._.H.e|y.,._.w.o.....
#_._._._._|_._.B.L.A.C.K.B.E.A.R.D._|_.A.r.r.r.....
#_._._._._|_._._._._._._._._._._.W.o|_.i.s.t._.....
# Sometimes the speaker of a line is too long and pdfplumber doesnt visualise the table column correctly
# this leads to 'Blackbeards' name to overlap with the text line of the previous line
# SPEAKER_NAME_WINDOW is the start and end index where the program should look for the character name
# this should be as tiny and on point as possible
# the window should start in the whitespace area before the speaker name, but can end in the text area after the speaker

In [49]:
TEST_SPEAKER_NAME_WINDOW = [5, 18]
print('       JUSTUS   Hey, wo'[TEST_SPEAKER_NAME_WINDOW[0]:TEST_SPEAKER_NAME_WINDOW[1]+1])
print('       BLACKBEARD  Arrr'[TEST_SPEAKER_NAME_WINDOW[0]:TEST_SPEAKER_NAME_WINDOW[1]+1])
print('                Wo ist '[TEST_SPEAKER_NAME_WINDOW[0]:TEST_SPEAKER_NAME_WINDOW[1]+1])

  JUSTUS   Hey
  BLACKBEARD  
           Wo 


## Set Text Start Index

In [50]:
TEXT_START_INDEX = 20
# this is the index of the char where the speakers text already started
# this should be BEFORE the text starts

#SPEAKER_NAME_WINDOW = [5, 18]
#_._._._._|_._.J.U.S.T.U.S._._._.H.e|y.,._.w.o.....
#_._._._._|_._.B.L.A.C.K.B.E.A.R.D._|_.A.r.r.r.....
#_._._._._|_._._._._._._._._._._.W.o|_.i.s.t._.....
#                           ^
#     TEXT START INDEX      |

## Set Speaker Name Window Threshold
- look at the defined speaker name window of a text line
- count the number of whitespace and non-whitespace chararcters
- divide to get a percentage
- below threshold -> probably no speaker, just remnants of the text
- above threshold -> probably a speaker with some whitespaces at the start (remember to set the window wider at the start)

In [51]:
SPEAKER_NAME_WINDOW_THRESHOLD = 0.5

## Example for page 1

In [52]:
page_text = pdf.pages[0].extract_text(layout=True)
print(page_text)

                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                 Die  drei  ???                                   
                          und  der  Fluch  des  Rubins                            
                                                                                  
                                                                                  
                                                                                  
                               Hörspielskript von Astrid                          
    

## Concat all pages

In [53]:
extracted_raw_text = ''
for page in pdf.pages:
    page_text = page.extract_text(layout=True)
    extracted_raw_text += page_text
print(extracted_raw_text)

                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                 Die  drei  ???                                   
                          und  der  Fluch  des  Rubins                            
                                                                                  
                                                                                  
                                                                                  
                               Hörspielskript von Astrid                          
    

## Spit texts to individual lines

In [54]:
extracted_lines = extracted_raw_text.split('\n')
len(extracted_lines)

1513

## Algorithm

In [55]:
# saves current speaker, text pair to csv_rows
def save_to_rows():
    global current_speaker, current_speaker_lines
    if current_speaker is not None:
        # join the current speaker lines to one text
        text = ' '.join([line for line in current_speaker_lines if line is not None and line != ''])
        # append it to csv
        csv_rows.append((current_speaker, text))
    # reset
    current_speaker = None
    current_speaker_lines = []

In [70]:
def speaker_and_text_line():
    pass

def only_text_line(line: str):
    global current_speaker, current_speaker_lines
    text = line.strip()
    if 'Lastwagenknattern' in text:
        print('FOUND')
    if current_speaker is None:
        current_speaker = 'REGIE'
        current_speaker_lines = [text]
    else:
        current_speaker_lines.append(text)

In [72]:
csv_rows = []  # rows for final csv
current_speaker = None  # currently speaking
current_speaker_lines = []  # lines attributed to current speaker

for line in extracted_lines:

    if line.strip() == '' or 'www.rocky-beach.com' in line:
        # empty row or credit row -> save
        save_to_rows()
        continue

    # try to find speaker in lookup table
    speaker_name_window_text = line[SPEAKER_NAME_WINDOW[0]:SPEAKER_NAME_WINDOW[1]+1]
    if len(speaker_name_window_text.strip()) == 0:
        only_text_line(line)
        continue
    
    whitespace_percentage_for_speaker_window = len(speaker_name_window_text.strip()) / len(speaker_name_window_text)
    
    if whitespace_percentage_for_speaker_window < SPEAKER_NAME_WINDOW_THRESHOLD:
        only_text_line(line)
        continue
    
    speaker_name, speaker_name_last_idx = lookup_speaker_name(speaker_name_window_text)

    if speaker_name is not None:
        # Speaker found, this is a Speaker AND Text line
        text = line[SPEAKER_NAME_WINDOW[0]+speaker_name_last_idx+1:].strip()
        if current_speaker is not None:
            # a new speaker is starting to say something, save the old and start new
            save_to_rows()
        current_speaker = speaker_name.strip()
        current_speaker_lines = [text] if text else []
    else:
        print(line)  # this should print lines where there should be a character but we found non in the lookup table
        # We detected a line with no Speaker and only Text
        only_text_line(line)

save_to_rows()

FOUND


## Save to CSV

In [69]:
with open(PATH_TO_OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(['Speaker', 'Text'])
    writer.writerows(csv_rows)